# LAB | Feature Engineering

In [139]:
#Libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

from sklearn.preprocessing import MinMaxScaler, StandardScaler

SEED=1

spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [140]:
spaceship.shape

(8693, 14)

**Check for data types**

In [141]:
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [142]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [143]:
spaceship = spaceship.dropna()

In [144]:
spaceship.isnull().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [145]:
spaceship['Cabin'].unique()

array(['B/0/P', 'F/0/S', 'A/0/S', ..., 'G/1499/S', 'G/1500/S', 'E/608/S'],
      shape=(5305,), dtype=object)

In [147]:
spaceship.loc[:,'Cabin'] = spaceship['Cabin'].str[0]

In [148]:
spaceship['Cabin'].unique()

array(['B', 'F', 'A', 'G', 'E', 'C', 'D', 'T'], dtype=object)

- Drop PassengerId and Name

In [149]:
spaceship = spaceship.drop (['PassengerId', 'Name'], axis=1)

In [150]:
spaceship.columns

Index(['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP',
       'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Transported'],
      dtype='object')

- For non-numerical columns, do dummies.

In [90]:
spaceship = pd.get_dummies(spaceship, drop_first=True)

In [91]:
spaceship.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,True,False,False,False,False,False,False,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,False,False,False,False,False,False,False,False,False,True,True
3,33.0,0.0,1283.0,371.0,3329.0,193.0,False,True,False,False,False,False,False,False,False,False,False,False,True,False
4,16.0,303.0,70.0,151.0,565.0,2.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False


In [152]:
#Checking the target values

print(spaceship['Transported'].value_counts())

Transported
True     3327
False    3279
Name: count, dtype: int64


In [153]:
#Transform False and True to 0 and 1:

spaceship['Transported'] = spaceship['Transported'].astype(int)

In [158]:
#Check if the 0 and 1 meaning is right:
print(spaceship['Transported'].value_counts())

Transported
1    3327
0    3279
Name: count, dtype: int64


In [156]:
# Now, let´s define x (features) and y (target)
features = spaceship.drop(columns = ["Transported"])
target = spaceship["Transported"]

**Perform Train Test Split**

In [125]:
# Use stratify, beacuse the y is a categorical target

X_train, X_test, y_train, y_test = train_test_split(
    features, target,
    test_size=0.2,
    random_state=SEED,
    stratify=y)

print(X_train.shape)
print(X_test.shape)

(5284, 19)
(1322, 19)


In [161]:
# FEATURE ENGENEERING: Feature Scaling
# Standardize with MinMaxScaler

scaler = MinMaxScaler()
scaler.set_output(transform="pandas")  # mantém nomes das colunas
# set_output(transform="pandas") makes the scaler return a DataFrame (keeps the real column names and index)

# Fit ONLY on X_train --> avoids leaking any information from the test set into the scaling parameters
X_train_scaled_df = scaler.fit_transform(X_train)

# Then apply that same transformation to X_test
X_test_scaled_df = scaler.transform(X_test)

# BEFORE: original scale
print('BEFORE scaling (X_train):')
print(X_train.describe().loc[['mean', 'std']].round(4))
print()

# AFTER: standardized scale
print('AFTER scaling (X_train):')
print(X_train_scaled_df.describe().loc[['mean', 'std']].round(4))

BEFORE scaling (X_train):
          Age  RoomService  FoodCourt  ShoppingMall        Spa     VRDeck
mean  28.9080     222.4037   475.6171      176.9722   308.2684   296.1684
std   14.4869     639.4044  1620.5749      592.2009  1108.4575  1118.2181

AFTER scaling (X_train):
         Age  RoomService  FoodCourt  ShoppingMall     Spa  VRDeck  \
mean  0.3659       0.0224     0.0160        0.0144  0.0186  0.0146   
std   0.1834       0.0645     0.0544        0.0483  0.0668  0.0550   

      HomePlanet_Europa  HomePlanet_Mars  CryoSleep_True  Cabin_B  Cabin_C  \
mean             0.2561           0.2087          0.3556   0.0973   0.0893   
std              0.4365           0.4064          0.4787   0.2964   0.2852   

      Cabin_D  Cabin_E  Cabin_F  Cabin_G  Cabin_T  Destination_PSO J318.5-22  \
mean   0.0590   0.1016   0.3231   0.2986   0.0004                      0.092   
std    0.2357   0.3022   0.4677   0.4577   0.0195                      0.289   

      Destination_TRAPPIST-1e  VIP_True

**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [162]:
#Create the KNN Classifier Model (Classifier because we have a categorical target)

knn = KNeighborsClassifier(n_neighbors=10)

In [163]:
#Train the model

knn.fit(X_train_reduced, y_train);
print("Model is trained!")

Model is trained!


In [164]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_reduced, y_train)

y_pred_train = knn.predict(X_train_reduced)
y_pred_test = knn.predict(X_test_reduced)

print("Accuracy (Train):", accuracy_score(y_train, y_pred_train))
print("Accuracy (Test):", accuracy_score(y_test, y_pred_test))

print(classification_report(y_test, y_pred_test))

Accuracy (Train): 0.753217259651779
Accuracy (Test): 0.7072617246596067
              precision    recall  f1-score   support

       False       0.69      0.74      0.71       656
        True       0.72      0.68      0.70       666

    accuracy                           0.71      1322
   macro avg       0.71      0.71      0.71      1322
weighted avg       0.71      0.71      0.71      1322



- Evaluate your model's performance. Comment it

In [98]:
# Train accuracy ~75% — the model learns the patterns reasonably well.
# Test accuracy ~71% — performance is slightly lower on unseen data but still consistent.
# Small gap (≈4%) — indicates no overfitting.
# Balanced precision/recall — the model treats both classes fairly evenly.
# Overall — stable model with moderate performance and room for improvement.